# EOH Search for an Eight-Feature FEI Extractor

This notebook runs a controlled EOH search in which every candidate feature-extraction function must return **exactly eight scalar features**.

The data split matches the MARS benchmark:

- 84 training images;
- 28 validation images;
- 28 held-out test images.

The validation set supplies EOH fitness. The test set remains untouched until a final program has been selected.


## 1. Imports and configuration


In [1]:
from pathlib import Path
import ast
import importlib
import inspect
import os
import sys
import time

import numpy as np
from skimage import color, io, transform
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd()
FEI_DIR = Path.home() / "Downloads" / "originalimages_part1"
EOH_PROBLEM_PARENT = Path.home() / "Downloads"

IMAGE_SIZE = (64, 64)
SELECTED_SUBJECTS = tuple(range(1, 11))
RANDOM_STATE = 42

# These values produce the MARS benchmark split: 84 / 28 / 28.
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_DEVELOPMENT = 0.25

MAX_SECONDS_PER_EVALUATION = 150.0
N_FEATURES = 8

print("Project folder:", PROJECT_ROOT)
print("FEI folder:", FEI_DIR)
print("FEI folder exists:", FEI_DIR.exists())
print("EOH problem folder:", EOH_PROBLEM_PARENT)

if not FEI_DIR.exists():
    raise FileNotFoundError(
        f"FEI dataset folder not found:\n{FEI_DIR}"
    )


Project folder: c:\Users\james\Downloads
FEI folder: <USER_HOME>\Downloads\originalimages_part1
FEI folder exists: True
EOH problem folder: <USER_HOME>\Downloads


## 2. Verify the EOH installation


In [2]:
import eoh
from eoh import EoH, LLMConfig

print("EOH imported from:")
print(Path(inspect.getfile(eoh)).resolve())
print("EoH signature:", inspect.signature(EoH))
print("LLMConfig signature:", inspect.signature(LLMConfig))


EOH imported from:
<USER_HOME>\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\eoh\__init__.py
EoH signature: (llm: eoh.config.LLMConfig, problem, pop_size: int = 5, n_pop: int = 20, operators: list = None, operator_weights: list = None, n_parents: int = 2, num_samplers: int = 1, num_evaluators: int = 1, max_sample_nums: int = None, n_processes: int = None, output_dir: str = './', debug: bool = False, use_seed: bool = False, seed_path: str = './seeds/seeds.json', use_continue: bool = False, continue_path: str = './results/pops/population_generation_0.json', continue_id: int = 0)
LLMConfig signature: (api_endpoint: str = None, api_key: str = None, model: str = None, use_local: bool = False, local_url: str = None, timeout: int = 180) -> None


## 3. Load and preprocess the first ten FEI subjects


In [3]:
def preprocess_image(image_path, image_size=IMAGE_SIZE):
    image = io.imread(image_path)

    if image.ndim == 3:
        image = color.rgb2gray(image)

    image = transform.resize(
        image,
        image_size,
        anti_aliasing=True,
        preserve_range=False,
    )

    return np.clip(
        np.asarray(image, dtype=np.float32),
        0.0,
        1.0,
    )


def infer_fei_subject_id(image_path):
    try:
        return int(image_path.stem.split("-")[0])
    except ValueError:
        return None


image_paths = sorted(
    list(FEI_DIR.glob("*.jpg"))
    + list(FEI_DIR.glob("*.jpeg"))
    + list(FEI_DIR.glob("*.png"))
)

images = []
labels = []
used_paths = []

for image_path in image_paths:
    subject_id = infer_fei_subject_id(image_path)

    if subject_id not in SELECTED_SUBJECTS:
        continue

    images.append(preprocess_image(image_path))
    labels.append(subject_id - 1)
    used_paths.append(image_path)

if not images:
    raise RuntimeError("No selected FEI images were loaded.")

X_images = np.stack(images).astype(np.float32)
y = np.asarray(labels, dtype=np.int64)

print("Image array shape:", X_images.shape)
print("Label array shape:", y.shape)
print("Classes:", np.unique(y))
print("Pixel range:", float(X_images.min()), "to", float(X_images.max()))

assert X_images.shape == (140, 64, 64)
assert y.shape == (140,)
assert set(np.unique(y)) == set(range(10))


Image array shape: (140, 64, 64)
Label array shape: (140,)
Classes: [0 1 2 3 4 5 6 7 8 9]
Pixel range: 0.0052183386869728565 to 0.824233889579773


## 4. Create the benchmark-compatible split


In [4]:
X_development, X_test, y_development, y_test = train_test_split(
    X_images,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=VALIDATION_SIZE_WITHIN_DEVELOPMENT,
    random_state=RANDOM_STATE,
    stratify=y_development,
)

# Optional aliases matching the MARS benchmark notebook.
X_val = X_validation
y_val = y_validation

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Test:", X_test.shape, y_test.shape)

assert X_train.shape == (84, 64, 64)
assert X_validation.shape == (28, 64, 64)
assert X_test.shape == (28, 64, 64)

assert len(X_train) + len(X_validation) + len(X_test) == len(X_images)
assert set(np.unique(y_train)) == set(np.unique(y))
assert set(np.unique(y_validation)) == set(np.unique(y))
assert set(np.unique(y_test)) == set(np.unique(y))

print("Benchmark split checks passed.")


Training: (84, 64, 64) (84,)
Validation: (28, 64, 64) (28,)
Test: (28, 64, 64) (28,)
Benchmark split checks passed.


## 5. Import and construct the fixed eight-feature EOH problem

The module `fei_eoh_problem_8.py` must be stored in the user's Downloads folder.


In [5]:
if str(EOH_PROBLEM_PARENT) not in sys.path:
    sys.path.insert(0, str(EOH_PROBLEM_PARENT))

if "fei_eoh_problem_8" in sys.modules:
    del sys.modules["fei_eoh_problem_8"]

importlib.invalidate_caches()

from fei_eoh_problem_8 import FEIFeatureExtractionProblem8

eoh8_problem = FEIFeatureExtractionProblem8(
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    timeout=180,
    n_processes=1,
    max_seconds_per_evaluation=MAX_SECONDS_PER_EVALUATION,
)

print("Problem module:", sys.modules["fei_eoh_problem_8"].__file__)
print("Problem class:", type(eoh8_problem).__name__)
print("Required feature dimension:", eoh8_problem.N_FEATURES)
print("Training images:", eoh8_problem.X_train.shape)
print("Validation images:", eoh8_problem.X_validation.shape)

assert eoh8_problem.N_FEATURES == 8
assert eoh8_problem.X_train.shape == (84, 64, 64)
assert eoh8_problem.X_validation.shape == (28, 64, 64)


Problem module: <USER_HOME>\Downloads\fei_eoh_problem_8.py
Problem class: FEIFeatureExtractionProblem8
Required feature dimension: 8
Training images: (84, 64, 64)
Validation images: (28, 64, 64)


## 6. Local eight-feature preflight

These tests do not contact the LLM API.


In [6]:
template_namespace = {"np": np}

exec(
    compile(
        eoh8_problem.template_program,
        "<eoh8_template>",
        "exec",
    ),
    template_namespace,
)

template_function = template_namespace["extract_features"]

template_vector = np.asarray(
    template_function(X_train[0]),
    dtype=np.float64,
).reshape(-1)

template_fitness = eoh8_problem.evaluate_program(
    eoh8_problem.template_program,
    template_function,
)

print("Template output shape:", template_vector.shape)
print("Template fitness:", template_fitness)

assert template_vector.shape == (8,)
assert np.all(np.isfinite(template_vector))
assert isinstance(template_fitness, float)
assert np.isfinite(template_fitness)

template_validation_accuracy = 1.0 - template_fitness

print(
    "Template validation accuracy:",
    f"{template_validation_accuracy * 100:.2f}%",
)


Template output shape: (8,)
Template fitness: 0.1428571428571429
Template validation accuracy: 85.71%


In [7]:
def invalid_nine_feature_function(image):
    image = np.asarray(image, dtype=float)

    return np.array(
        [
            np.mean(image),
            np.std(image),
            np.min(image),
            np.max(image),
            np.median(image),
            np.percentile(image, 25),
            np.percentile(image, 75),
            np.mean(np.abs(np.diff(image, axis=0))),
            np.mean(np.abs(np.diff(image, axis=1))),
        ],
        dtype=float,
    )


invalid_fitness = eoh8_problem.evaluate_program(
    "Invalid test candidate returning nine features.",
    invalid_nine_feature_function,
)

print("Nine-feature candidate fitness:", invalid_fitness)

assert invalid_fitness is None
print("PASS: candidates not returning exactly eight features are rejected.")


Nine-feature candidate fitness: None
PASS: candidates not returning exactly eight features are rejected.


## 7. Configure DeepSeek securely

The API key must be available as the environment variable `DEEPSEEK_API_KEY`. No key is written into this notebook.


In [8]:
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

if not deepseek_api_key:
    raise RuntimeError(
        "DEEPSEEK_API_KEY is not available in this notebook session."
    )

llm_config = LLMConfig(
    api_endpoint="api.deepseek.com",
    api_key=deepseek_api_key,
    model="deepseek-v4-flash",
    timeout=180,
)

print("LLM configuration created")
print("Endpoint:", llm_config.api_endpoint)
print("Model:", llm_config.model)
print("Timeout:", llm_config.timeout)
print("API key loaded:", bool(llm_config.api_key))


LLM configuration created
Endpoint: api.deepseek.com
Model: deepseek-v4-flash
Timeout: 180
API key loaded: True


## 8. Configure the minimal EOH-8 runner


In [9]:
eoh8_runner = EoH(
    llm=llm_config,
    problem=eoh8_problem,
    pop_size=2,
    n_pop=1,
    operators=["e1"],
    num_samplers=1,
    num_evaluators=1,
)

print("Minimal EOH-8 runner configured")
print("Population size:", eoh8_runner._config.pop_size)
print("Populations:", eoh8_runner._config.n_pop)
print("Operators:", eoh8_runner._config.operators)
print(
    "Runner uses EOH-8 problem:",
    eoh8_runner._problem is eoh8_problem,
)
print("No API request has been made.")


Minimal EOH-8 runner configured
Population size: 2
Populations: 1
Operators: ['e1']
Runner uses EOH-8 problem: True
No API request has been made.


## 9. Final local preflight


In [10]:
checks = {
    "API key loaded":
        bool(os.getenv("DEEPSEEK_API_KEY")),

    "Correct endpoint":
        eoh8_runner._config.llm.api_endpoint == "api.deepseek.com",

    "Model configured":
        bool(eoh8_runner._config.llm.model),

    "Population size is 2":
        eoh8_runner._config.pop_size == 2,

    "One population":
        eoh8_runner._config.n_pop == 1,

    "Only E1 enabled":
        eoh8_runner._config.operators == ["e1"],

    "Eight-feature problem attached":
        eoh8_runner._problem is eoh8_problem,

    "Required dimension is 8":
        eoh8_problem.N_FEATURES == 8,

    "Template program parses":
        bool(ast.parse(eoh8_problem.template_program)),

    "Problem class importable":
        FEIFeatureExtractionProblem8.__module__
        == "fei_eoh_problem_8",

    "Run method exists":
        callable(getattr(eoh8_runner, "run", None)),

    "Template candidate valid":
        isinstance(template_fitness, float)
        and np.isfinite(template_fitness),

    "Nine-feature candidate rejected":
        invalid_fitness is None,
}

print("EOH-8 FINAL LOCAL PREFLIGHT")
print("-" * 58)

for name, passed in checks.items():
    print(f"{name:<38} {'PASS' if passed else 'FAIL'}")

failed_checks = [
    name for name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise RuntimeError(
        "Do not start the API run. Failed checks: "
        + ", ".join(failed_checks)
    )

print("-" * 58)
print("All local checks passed.")
print("The paid search remains disabled below.")


EOH-8 FINAL LOCAL PREFLIGHT
----------------------------------------------------------
API key loaded                         PASS
Correct endpoint                       PASS
Model configured                       PASS
Population size is 2                   PASS
One population                         PASS
Only E1 enabled                        PASS
Eight-feature problem attached         PASS
Required dimension is 8                PASS
Template program parses                PASS
Problem class importable               PASS
Run method exists                      PASS
Template candidate valid               PASS
Nine-feature candidate rejected        PASS
----------------------------------------------------------
All local checks passed.
The paid search remains disabled below.


## 10. Paid EOH-8 smoke test — disabled by default

Changing the flag to `True` will make API requests and may incur charges.


In [11]:
RUN_EOH8_PAID_SMOKE_TEST = True

eoh8_result = None
eoh8_runtime_seconds = None

if RUN_EOH8_PAID_SMOKE_TEST:
    print("Starting paid EOH-8 smoke test...")

    started = time.perf_counter()
    eoh8_result = eoh8_runner.run()
    eoh8_runtime_seconds = time.perf_counter() - started

    print("EOH-8 smoke test completed.")
    print("Runtime:", f"{eoh8_runtime_seconds:.2f} seconds")
    print("Result type:", type(eoh8_result).__name__)
    print("Result preview:", repr(eoh8_result)[:3000])
else:
    print("EOH-8 paid smoke test not started.")
    print("No API request was made.")


Starting paid EOH-8 smoke test...
[2026-08-03 12:16:40] LLM: deepseek-v4-flash @ api.deepseek.com


[2026-08-03 12:16:44] LLM connection verified.
[2026-08-03 12:16:44] ======================================================
[2026-08-03 12:16:44]   EoH
[2026-08-03 12:16:44]   LLM      : deepseek-v4-flash @ api.deepseek.com
[2026-08-03 12:16:44]   EC       : gen=1  pop=2  ops=[e1]
[2026-08-03 12:16:44]   Sampling : init=4 (2×pop)  evo_budget=2
[2026-08-03 12:16:44]   Pipeline : samplers=1  evaluators=1 (async)
[2026-08-03 12:16:44]   Timeout  : llm=180s  eval=180s
[2026-08-03 12:16:44] ======================================================
[2026-08-03 12:16:44] 
[Init]  (4 samples → pop=2)  samplers=1  evaluators=1
[2026-08-03 12:20:06]   #1    [i1]  0.32143           best=0.32143  *
[2026-08-03 12:22:10]   #2    [i1]  0.5               best=0.32143
[2026-08-03 12:24:38]   #3    [i1]  0.25              best=0.25  *
[2026-08-03 12:26:55]   #4    [i1]  0.25              best=0.25
[2026-08-03 12:26:55]   Init done: 4/4 evaluated  pop=2  best=0.25  elapsed=10.2m
[2026-08-03 12:26:55] 
[Evo

## 11. Inspect generated files after a paid run


In [17]:
candidate_roots = [
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "outputs",
    Path.cwd(),
]

recent_candidate_files = []

for root in candidate_roots:
    if not root.exists():
        continue

    for pattern in ("*.py", "*.txt", "*.json", "*.csv"):
        recent_candidate_files.extend(root.rglob(pattern))

recent_candidate_files = sorted(
    set(recent_candidate_files),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

print("Most recently modified candidate/result files:")

for path in recent_candidate_files[:20]:
    print(path.resolve())


Most recently modified candidate/result files:
<USER_HOME>\Downloads\results\run_log.txt
<USER_HOME>\Downloads\results\samples\samples_1~6.json
<USER_HOME>\Downloads\results\pops_best\population_generation_1.json
<USER_HOME>\Downloads\results\pops\population_generation_1.json
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_123042_068176_11688_success.txt
<USER_HOME>\Downloads\results\samples\samples_best.json
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_122825_108512_21132_success.txt
<USER_HOME>\Downloads\results\pops\population_generation_0.json
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_122655_408715_28108_success.txt
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_122437_996615_23416_success.txt
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_122210_047881_3112_success.txt
<USER_HOME>\Downloads\results\fei_eoh_8\candidate_diagnostics\20260803_122006_539529_1318

In [18]:
from pathlib import Path
import re

diagnostics_dir = (
    Path.cwd()
    / "results"
    / "fei_eoh_8"
    / "candidate_diagnostics"
)

success_files = sorted(
    diagnostics_dir.glob("*_success.txt"),
    key=lambda path: path.stat().st_mtime,
)

candidates = []

for path in success_files:
    text = path.read_text(encoding="utf-8")

    fitness_match = re.search(
        r"fitness=([0-9.]+)",
        text,
    )

    accuracy_match = re.search(
        r"accuracy=([0-9.]+)",
        text,
    )

    if fitness_match:
        candidates.append(
            {
                "path": path,
                "fitness": float(fitness_match.group(1)),
                "accuracy": (
                    float(accuracy_match.group(1))
                    if accuracy_match
                    else None
                ),
                "text": text,
            }
        )

if not candidates:
    raise RuntimeError(
        "No successful EOH candidate diagnostics were found."
    )

best_candidate = min(
    candidates,
    key=lambda item: item["fitness"],
)

print("Best diagnostic file:")
print(best_candidate["path"])

print("\nBest fitness:")
print(best_candidate["fitness"])

print("\nValidation accuracy:")
print(best_candidate["accuracy"])

print("\nFull saved diagnostic:")
print(best_candidate["text"])

Best diagnostic file:
c:\Users\james\Downloads\results\fei_eoh_8\candidate_diagnostics\20260801_000718_839151_27880_success.txt

Best fitness:
0.071429

Validation accuracy:
0.928571

Full saved diagnostic:
STATUS: success

MESSAGE:
feature_dim=8, accuracy=0.928571, fitness=0.071429, runtime=0.022s

PROGRAM:
import numpy as np

def extract_features(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=float)
    h, w = image.shape
    eps = 1e-12

    half = w // 2
    left = image[:, :half]
    right = np.fliplr(image[:, -half:])
    symmetry = np.mean(np.abs(left - right))

    eye_band = image[int(0.25*h):int(0.45*h), int(0.15*w):int(0.85*w)]
    mouth_band = image[int(0.55*h):int(0.75*h), int(0.20*w):int(0.80*w)]

    dy, dx = np.gradient(image)
    mag = np.sqrt(dx*dx + dy*dy)
    edge_strength = np.mean(mag)
    edge_density = np.mean(mag > np.mean(mag) + np.std(mag))

    rows = np.arange(h).reshape(-1, 1)
    total = np.sum(image) + eps
    vertical_centroid = n

In [19]:
# -------------------------------------------------------
# Extract the complete winning EOH program
# -------------------------------------------------------

diagnostic_text = best_candidate["text"]

if "PROGRAM:" not in diagnostic_text:
    raise RuntimeError(
        "The best diagnostic file does not contain a PROGRAM section."
    )

BEST_PROGRAM_TEXT = diagnostic_text.split(
    "PROGRAM:",
    maxsplit=1,
)[1].strip()

print("Complete winning program extracted.")
print("Program length:", len(BEST_PROGRAM_TEXT), "characters")
print()
print(BEST_PROGRAM_TEXT)

Complete winning program extracted.
Program length: 959 characters

import numpy as np

def extract_features(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=float)
    h, w = image.shape
    eps = 1e-12

    half = w // 2
    left = image[:, :half]
    right = np.fliplr(image[:, -half:])
    symmetry = np.mean(np.abs(left - right))

    eye_band = image[int(0.25*h):int(0.45*h), int(0.15*w):int(0.85*w)]
    mouth_band = image[int(0.55*h):int(0.75*h), int(0.20*w):int(0.80*w)]

    dy, dx = np.gradient(image)
    mag = np.sqrt(dx*dx + dy*dy)
    edge_strength = np.mean(mag)
    edge_density = np.mean(mag > np.mean(mag) + np.std(mag))

    rows = np.arange(h).reshape(-1, 1)
    total = np.sum(image) + eps
    vertical_centroid = np.sum(image * rows) / (total * h)

    return np.array([
        np.mean(image),
        np.std(image),
        np.mean(eye_band),
        np.mean(mouth_band),
        symmetry,
        edge_strength,
        edge_density,
        vertical_ce

## 12. Evaluate a selected program on validation data

Paste a selected EOH-generated program below only after inspecting it. This cell still does **not** use the held-out test set.


In [20]:
BEST_PROGRAM_TEXT = """
# Paste the selected EOH-generated program here.
"""

best_program_text = BEST_PROGRAM_TEXT.strip()
best_evolved_function = None
selected_validation_accuracy = None

if "def extract_features" in best_program_text:
    namespace = {"np": np}

    exec(
        compile(
            best_program_text,
            "<best_eoh8_program>",
            "exec",
        ),
        namespace,
    )

    best_evolved_function = namespace.get("extract_features")

    if not callable(best_evolved_function):
        raise RuntimeError(
            "The supplied program does not define extract_features."
        )

    selected_fitness = eoh8_problem.evaluate_program(
        best_program_text,
        best_evolved_function,
    )

    if selected_fitness is None:
        raise RuntimeError(
            "The selected program failed the eight-feature evaluator."
        )

    selected_validation_accuracy = 1.0 - selected_fitness

    print(
        "Selected program validation accuracy:",
        f"{selected_validation_accuracy * 100:.2f}%",
    )
else:
    print("No selected EOH-8 program has been supplied yet.")


No selected EOH-8 program has been supplied yet.


## 13. Final held-out test evaluation — disabled by default

Use the test set only after the final program has been selected using validation performance.


In [21]:
RUN_FINAL_TEST_EVALUATION = False

if RUN_FINAL_TEST_EVALUATION:
    if not callable(best_evolved_function):
        raise RuntimeError(
            "Supply and validate the selected program before testing."
        )

    development_images = np.concatenate(
        [X_train, X_validation],
        axis=0,
    )
    development_labels = np.concatenate(
        [y_train, y_validation],
        axis=0,
    )

    development_features = eoh8_problem._build_feature_matrix(
        development_images,
        best_evolved_function,
    )
    test_features = eoh8_problem._build_feature_matrix(
        X_test,
        best_evolved_function,
    )

    classifier = make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=1.0,
            random_state=RANDOM_STATE,
            dual="auto",
            max_iter=10000,
        ),
    )

    classifier.fit(
        development_features,
        development_labels,
    )
    test_predictions = classifier.predict(test_features)
    final_test_accuracy = accuracy_score(
        y_test,
        test_predictions,
    )

    print(
        "Final held-out test accuracy:",
        f"{final_test_accuracy * 100:.2f}%",
    )
else:
    print("Final test evaluation is disabled.")
    print("The held-out test set remains untouched.")


Final test evaluation is disabled.
The held-out test set remains untouched.


## Controlled comparison status

At this stage:

- the handcrafted benchmark uses 8 features;
- EOH candidates are forced to use exactly 8 features;
- both use the same 84/28/28 benchmark split;
- the true test set remains reserved;
- the first paid run is deliberately disabled until all preflight checks pass.
